# Признаки МО из растров

Глобальные растры покрывают всю страну единой методикой, поэтому дополняют два других источника там, где те слабы: OSM — в сельской местности, где разметки мало, Росстат — там, где показатель скрыт по конфиденциальности.

| Растр | Что даёт | Зачем для кластеризации |
|---|---|---|
| Ночная освещённость VIIRS | яркость, сумма света | косвенная мера экономической активности |
| ESA WorldCover или GHSL | доли застройки, пашни, леса, воды | тип территории: городская, аграрная, лесная |
| Сетка населения WorldPop | население по пикселам | население там, где Росстат его скрыл |
| Рельеф | высота и её разброс | горы ограничивают застройку и транспорт |
| Климат WorldClim | температура, осадки | объясняет часть трат на транспорт и отопление |

**Файлы скачиваются вручную** и кладутся в `data/rasters/`. Ссылки — в `rasters.RECOMMENDED`. Автоматическую загрузку не делаем: у источников разные условия доступа, а файлы занимают от сотен мегабайт до десятков гигабайт.

**Результат** — `results/mo_raster_features.csv`, строка на МО, ключ `oktmo`. Склеивается с остальными признаками по тому же ключу.

**Где зональная статистика может ошибаться:** Эти случаи обрабатываются в `datasets/rasters.py`, а отчёт по каждому растру показывает, сколько МО попало в каждый:
* у растра своя система координат, поэтому перепроецируются границы, а не растр;
* `nodata` отбрасывается, иначе в среднее попадут значения вроде −9999;
* в мелкий городской округ может не попасть ни один центр пиксела грубого растра — для таких расчёт повторяется по касанию пикселов;
* огромные районы читаются прорежённо, из-за чего сумма становится приближённой, а среднее и доли почти не страдают.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from datasets import boundaries, rasters

DATA = Path('data')
OUT = Path('results'); OUT.mkdir(exist_ok=True)

BOUNDARIES_PATH = DATA / 'Municipalities_Russia_2025.geojson'
SOURCE_CRS = 3857                 # система координат файла границ
RASTER_DIR = DATA / 'rasters'
MAX_PIXELS = 4_000_000            # предел на одно МО, выше идёт прореживание чтения

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

## 1. Что скачать

In [2]:
display(pd.DataFrame(rasters.RECOMMENDED).T[['name', 'kind', 'why', 'url']])

,name,kind,why,url
nightlights,"VIIRS, годовая ночная освещённость",continuous,Косвенная мера экономической активности: освещ...,https://eogdata.mines.edu/products/vnl/
worldcover,"ESA WorldCover, типы земель 10 м",categorical,"Доли застройки, пашни, леса и воды описывают т...",https://esa-worldcover.org/en/data-access
ghsl_built,"GHSL, доля застроенной поверхности",continuous,Плотность застройки: отличает город от сельско...,https://human-settlement.emergency.copernicus....
population,"WorldPop, сетка населения 100 м",continuous,"Население там, где показатель Росстата скрыт п...",https://hub.worldpop.org/geodata/listing?id=29
elevation,Рельеф (GMTED2010 или Copernicus DEM),continuous,Высота и её разброс: горная территория огранич...,https://www.usgs.gov/coastal-changes-and-impac...
climate,"WorldClim, температура и осадки",continuous,"Климат объясняет часть трат на транспорт, отоп...",https://worldclim.org/data/worldclim21.html


In [ ]:
files = rasters.list_rasters(RASTER_DIR)

## 2. Границы МО

Берём тот же файл и ту же функцию, что и в `collect_features.ipynb`, чтобы ключ `oktmo` совпадал.

In [ ]:
mo_boundaries = boundaries.load(BOUNDARIES_PATH, crs=SOURCE_CRS)
display(mo_boundaries.head(3))

## 3. Настройка растров

Нужно прописать имена своих файлов. Обрабатываются только те, что лежат в папке, остальные пропускаются, поэтому ноутбук можно запускать по мере скачивания.

`scale` и `offset` нужны растрам, где записаны масштабированные целые: значение = сырое × `scale` + `offset`

In [ ]:
RASTERS = {
    'ntl':   dict(file='viirs_annual_2023.tif',        kind='continuous', prefix='ntl_'),
    'built': dict(file='GHS_BUILT_S_2020_100m.tif',    kind='continuous', prefix='built_'),
    'pop':   dict(file='rus_ppp_2020.tif',             kind='continuous', prefix='pop_'),
    'dem':   dict(file='gmted2010_rus.tif',            kind='continuous', prefix='dem_'),
    'tavg':  dict(file='wc2.1_30s_bio_1.tif',          kind='continuous', prefix='tavg_'),
    'land':  dict(file='worldcover_russia.vrt',        kind='categorical', prefix='land_',
                  classes=rasters.WORLDCOVER_CLASSES),
}

available = {name: cfg for name, cfg in RASTERS.items() if (RASTER_DIR / cfg['file']).exists()}
print('Найдены:', list(available) or 'ничего')
print('Отсутствуют:', [n for n in RASTERS if n not in available])

In [ ]:
for name, cfg in available.items():
    print(f'=== {name}: {cfg["file"]}')
    rasters.describe(RASTER_DIR / cfg['file'])
    print()

## 4. Расчёт

Самая долгая часть: для каждого растра перебираются все МО. Результат по каждому растру сохраняется отдельно, поэтому прерванный расчёт можно продолжить — готовые растры берутся из кэша.

In [ ]:
parts = {}
for name, cfg in available.items():
    cache = OUT / f'raster_{name}.csv'
    if cache.exists():
        parts[name] = pd.read_csv(cache, dtype={'oktmo': str}).set_index('oktmo')
        print(f'{name}: из кэша, {len(parts[name])} МО')
        continue
    part = rasters.zonal(
        RASTER_DIR / cfg['file'], mo_boundaries,
        kind=cfg['kind'], classes=cfg.get('classes'),
        scale=cfg.get('scale', 1.0), offset=cfg.get('offset', 0.0),
        band=cfg.get('band', 1), max_pixels=MAX_PIXELS, prefix=cfg['prefix'])
    part.to_csv(cache)
    parts[name] = part

## 5. Производные признаки

Сумма зависит от размера МО, поэтому переводим её в плотность и в расчёт на человека. Площадь берём из геометрии в равновеликой проекции, а не из числа пикселов: в градусной системе координат пиксел на севере уже, чем на юге.

Свет на человека — осмысленный признак сам по себе: он высок там, где много производства и транспорта при небольшом населении (нефтегазовые и промышленные районы), и низок в спальных территориях.

In [ ]:
area_km2 = (mo_boundaries.set_index('oktmo').to_crs(6933).area / 1e6).rename('area_km2')

features = pd.concat(parts.values(), axis=1) if parts else pd.DataFrame()
features = features.join(area_km2, how='outer')

if 'ntl_sum' in features:
    features['ntl_свет_на_км2'] = features['ntl_sum'] / features['area_km2']
if 'pop_sum' in features:
    features['pop_плотность_на_км2'] = features['pop_sum'] / features['area_km2']
if {'ntl_sum', 'pop_sum'} <= set(features.columns):
    features['ntl_свет_на_человека'] = features['ntl_sum'] / features['pop_sum'].replace(0, np.nan)
if 'built_mean' in features:
    features['built_доля_застройки'] = features['built_mean']

print(f'МО: {len(features)}, признаков: {features.shape[1]}')
display(features.describe().T[['count', 'mean', '50%', 'max']])

## 6. Проверка и сохранение

Смотрим на полноту и на правдоподобность. Две быстрые проверки здравым смыслом:
* МО с самой высокой освещённостью должны быть крупными городами;
* сумма населения по растру должна быть одного порядка с данными Росстата, иначе перепутаны единицы или взят не тот слой.

In [ ]:
coverage = features.notna().mean().sort_values()
print('Признаки с наименьшим покрытием:')
display(coverage.head(10).to_frame('доля МО со значением'))

if 'ntl_sum' in features:
    top = features.join(mo_boundaries.set_index('oktmo')['name'])
    print('МО с наибольшей суммарной освещённостью (ожидаем крупные города):')
    display(top.nlargest(10, 'ntl_sum')[['name', 'ntl_sum', 'ntl_mean', 'area_km2']])

if 'pop_sum' in features:
    print(f"Сумма населения по растру: {features['pop_sum'].sum():,.0f} человек".replace(',', ' '))

In [ ]:
features.index.name = 'oktmo'
features.to_csv(OUT / 'mo_raster_features.csv')
print('Сохранено:', OUT / 'mo_raster_features.csv')

## 7. Как склеить с остальным

Ключ тот же, что и у статичных признаков из `collect_features.ipynb`, — `oktmo`:

```python
static = pd.read_csv('results/mo_features.csv', dtype={'oktmo': str}).set_index('oktmo')
raster = pd.read_csv('results/mo_raster_features.csv', dtype={'oktmo': str}).set_index('oktmo')
all_features = static.join(raster, how='left')
```

Перед кластеризацией помнить, что сумма света, сумма населения и площадь сильно коррелируют с размером МО. В эксперименте 1 именно такие признаки увели кластеры в шкалу «крупный город — остальное», поэтому в модель лучше брать производные: свет на человека, свет на км², долю застройки, доли классов земель.